# Кейс 3. Продуктовые гипотезы. Шаг 0–1: загрузка данных и разведка

Ноутбук воспроизводим: данные скачиваются по публичной ссылке владельца, ничего не нужно загружать вручную.

**Как запускать:** `Среда выполнения → Выполнить все`. Первый запуск занимает примерно 30–50 минут (скачивание около 9,5 ГБ, конвертация в Parquet, разведка). Повторные запуски быстрее: Parquet сохраняется на Google Drive.

**Персональные данные.** При загрузке пароли, имена, токены, `raw_info`, данные карт и ссылки на оплату удаляются; IP, идентификаторы устройств и внешних аккаунтов заменяются хешем с секретной солью (соль хранится только на вашем Drive). В выводе ноутбука только агрегаты.

In [ ]:
# ---------- Настройки ----------
PUBLIC_URL = "https://disk.yandex.ru/d/UYuFNCrHvWR3DQ"
WORK = "/content/case3"                      # временная папка на диске Colab
USE_DRIVE = True                             # хранить Parquet и соль на Google Drive
DRIVE_DIR = "/content/drive/MyDrive/case3"   # папка проекта на Drive
LOG_SAMPLE_ZIPS = 60                         # сколько архивов логов просматривать целиком при поиске ключевых слов
SEED = 42

In [ ]:
import os, re, json, time, random, hashlib, secrets, zipfile, collections, urllib.parse, urllib.request
import pandas as pd

try:
    import duckdb
except ImportError:
    !pip -q install duckdb
    import duckdb

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
BASE = DRIVE_DIR if USE_DRIVE else WORK
PARQUET = f"{BASE}/parquet"
RAW = f"{WORK}/raw"
for d in (WORK, RAW, BASE, PARQUET):
    os.makedirs(d, exist_ok=True)

salt_path = f"{BASE}/salt.txt"            # секрет: не коммитить в репозиторий
if not os.path.exists(salt_path):
    with open(salt_path, "w") as f:
        f.write(secrets.token_hex(16))
SALT = open(salt_path).read().strip()
print("duckdb", duckdb.__version__, "| данные:", BASE)

## 1. Скачивание с Яндекс.Диска

CSV скачиваются, только если Parquet ещё не собран. `logs.zip` скачивается и распаковывается один раз за сеанс Colab (вложенные архивы остаются запакованными, их читаем напрямую).

In [ ]:
API = "https://cloud-api.yandex.net/v1/disk/public/resources"

def api_get(url, retries=5):
    for i in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return json.load(r)
        except Exception as e:
            print(f"  API: {e}, повтор через {2**i} с"); time.sleep(2 ** i)
    raise RuntimeError("Яндекс.Диск недоступен")

def list_dir(path):
    items, offset = [], 0
    while True:
        q = urllib.parse.urlencode({"public_key": PUBLIC_URL, "path": path, "limit": 1000, "offset": offset})
        page = api_get(f"{API}?{q}")["_embedded"]
        items += page["items"]; offset += len(page["items"])
        if offset >= page["total"] or not page["items"]:
            return items

def walk(path="/"):
    for it in list_dir(path):
        if it["type"] == "dir":
            yield from walk(it["path"])
        else:
            yield it

def download(item):
    target = os.path.join(RAW, item["path"].lstrip("/"))
    if os.path.exists(target) and os.path.getsize(target) == item.get("size"):
        return target
    os.makedirs(os.path.dirname(target), exist_ok=True)
    q = urllib.parse.urlencode({"public_key": PUBLIC_URL, "path": item["path"]})
    href = api_get(f"{API}/download?{q}")["href"]
    with urllib.request.urlopen(href, timeout=600) as r, open(target + ".part", "wb") as f:
        while chunk := r.read(8 << 20):
            f.write(chunk)
    os.replace(target + ".part", target)
    return target

remote = list(walk("/"))
print(f"На диске файлов: {len(remote)}, объём {sum(i.get('size', 0) for i in remote) / 2**30:.2f} ГБ")
parquet_ready = os.path.exists(f"{PARQUET}/_done.json")
for it in remote:
    is_logs = it["name"].lower() == "logs.zip"
    if not is_logs and parquet_ready:
        continue
    t0 = time.time(); download(it)
    print(f"  {it['path']} ({it.get('size', 0) / 2**20:.0f} МБ) за {time.time() - t0:.0f} с")

In [ ]:
def find_dirs():
    data_dir = struct_dir = logs_zip = None
    for cur, ds, fs in os.walk(RAW):
        if "users.csv" in fs:
            if os.path.basename(cur) == "db_structure":
                struct_dir = cur
            else:
                data_dir = cur
        for f in fs:
            if f.lower() == "logs.zip":
                logs_zip = os.path.join(cur, f)
    return data_dir, struct_dir, logs_zip

DATA_DIR, STRUCT_DIR, LOGS_ZIP = find_dirs()
print("CSV:", DATA_DIR, "\nструктура:", STRUCT_DIR, "\nлоги:", LOGS_ZIP)

LOGS_DIR = f"{WORK}/logs"
if LOGS_ZIP and not os.path.exists(f"{LOGS_DIR}/_done"):
    t0 = time.time()
    with zipfile.ZipFile(LOGS_ZIP) as z:
        z.extractall(LOGS_DIR)
    open(f"{LOGS_DIR}/_done", "w").close()
    print(f"logs.zip распакован за {time.time() - t0:.0f} с")

# корень логов: папка, где лежат числовые подпапки (номер = вероятно users.id)
LOG_ROOT = None
for cur, ds, fs in os.walk(LOGS_DIR):
    if sum(d.isdigit() for d in ds) > 10:
        LOG_ROOT = cur; break
print("корень логов:", LOG_ROOT)

## 2. Загрузка CSV в Parquet

В выгрузке у CSV **нет строки заголовков** (кроме `user_connection_logs_*`), поэтому имена колонок берутся из `db_structure`. Разделитель всегда `;`, JSON-поля в кавычках. Все поля читаются как текст; время приводится к `TIMESTAMP` (значения с поясом переводятся в UTC, значения без пояса остаются как есть, их пояс проверяем в разведке).

In [ ]:
TIME_COL = re.compile(r"(_at$|^last_seen$|tstamp|expired_time|block_start|block_end)")
TZ_RE = re.compile(r" [0-9:.]+[+-][0-9]{2}(:?[0-9]{2})?$")

# Правила обезличивания: drop — удалить, hash — хеш с солью, has — только флаг заполненности
PD_RULES = {
    "users": {"name": "drop", "email": "email_domain", "password": "drop", "remna_id": "hash",
              "remna_short_id": "hash", "dashamail_id": "drop", "ga4_client_id": "hash",
              "appmetrica_device_id": "hash", "appmetrica_profile_id": "hash"},
    "devices": {"device_identifier": "hash", "device_name": "drop",
                "fcm_push_token": "has", "hms_push_token": "has"},
    "sessions": {"ip_address": "hash"},
    "external_accounts": {"account_id": "hash", "raw_info": "drop"},
    "saved_payment_methods": {"payment_method_id": "drop", "card_info": "card"},
    "transactions": {"payment_link": "drop", "raw_response_payment": "payment_json"},
    "payment_providers": {"settings": "drop"},
}

try:
    duckdb.sql("SELECT sha256('a')").fetchone(); HASHFN = "sha256"
except Exception:
    HASHFN = "md5"   # старые версии DuckDB без sha256

def h(expr):
    return f"CASE WHEN {expr} IS NULL OR {expr} = '' THEN NULL ELSE {HASHFN}('{SALT}' || {expr}) END"

def j(path, r="r"):
    return f"json_extract_string({r}, '{path}')"

def select_list(table, cols):
    out = []
    for c in cols:
        q, rule = f'"{c}"', PD_RULES.get(table, {}).get(c)
        if rule == "drop":
            continue
        elif rule == "hash":
            out.append(f"{h(q)} AS {q}")
        elif rule == "has":
            out.append(f"({q} IS NOT NULL AND {q} <> '') AS \"has_{c}\"")
        elif rule == "email_domain":
            out.append(f"lower(split_part({q}, '@', 2)) AS email_domain")
        elif rule == "card":
            fp = (f"json_extract_string({q}, '$.first6') || ':' || json_extract_string({q}, '$.last4') || ':' || "
                  f"coalesce(json_extract_string({q}, '$.expiry_month'), '') || '/' || coalesce(json_extract_string({q}, '$.expiry_year'), '')")
            out.append(f"CASE WHEN json_valid({q}) AND json_extract_string({q}, '$.last4') IS NOT NULL "
                       f"THEN {HASHFN}('{SALT}' || {fp}) END AS card_fp")
            out.append(f"CASE WHEN json_valid({q}) THEN json_extract_string({q}, '$.card_type') END AS card_type")
            out.append(f"CASE WHEN json_valid({q}) THEN json_extract_string({q}, '$.issuer_country') END AS card_country")
        elif rule == "payment_json":
            r = f"(CASE WHEN json_valid({q}) THEN {q} END)"
            J = lambda p: j(p, r)
            fields = {
                "resp_keys": f"json_keys({r})",
                "resp_status": f"coalesce({J('$.status')}, {J('$.Status')})",
                "resp_paid": f"coalesce({J('$.paid')}, {J('$.Paid')})",
                "resp_currency": f"coalesce({J('$.amount.currency')}, {J('$.Amount.Currency')}, {J('$.Currency')})",
                "pay_method": f"coalesce({J('$.payment_method.type')}, {J('$.PaymentMethod')})",
                "yk_cancel_reason": J('$.cancellation_details.reason'),
                "yk_cancel_party": J('$.cancellation_details.party'),
                "other_reason": f"coalesce({J('$.Reason')}, {J('$.ReasonCode')}, {J('$.reason')}, {J('$.CardHolderMessage')})",
            }
            for name, expr in fields.items():
                out.append(f"{expr} AS {name}")
        elif TIME_COL.search(c):
            out.append(f"ts({q}) AS {q}")
        else:
            out.append(q)
    return ", ".join(out)

con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")
con.execute(f"SET temp_directory = '{WORK}/duck_tmp'")
con.execute("""CREATE OR REPLACE MACRO ts(x) AS CASE
    WHEN regexp_matches(x, ' [0-9:.]+[+-][0-9]{2}(:?[0-9]{2})?$') THEN timezone('UTC', try_cast(x AS TIMESTAMPTZ))
    ELSE try_cast(x AS TIMESTAMP) END""")

In [ ]:
def struct_cols(table):
    with open(os.path.join(STRUCT_DIR, f"{table}.csv"), encoding="utf-8-sig") as f:
        return [c.strip() for c in f.readline().strip().split(";")]

def read_csv_expr(path, cols, header):
    spec = "{" + ", ".join(f"'{c}': 'VARCHAR'" for c in cols) + "}"
    return (f"read_csv('{path}', delim=';', quote='\"', escape='\"', header={str(header).lower()}, "
            f"columns={spec}, auto_detect=false, store_rejects=true, max_line_size=50000000)")

def rejects():
    try:
        return con.sql("SELECT count(*) FROM reject_errors").fetchone()[0]
    except Exception:
        return 0

LOAD_LOG, TZ_INFO = [], {}
if not parquet_ready:
    for fname in sorted(os.listdir(DATA_DIR)):
        if not fname.lower().endswith(".csv"):
            continue
        path, stem = os.path.join(DATA_DIR, fname), fname[:-4]
        table = "user_connection_logs" if stem.startswith("user_connection_logs_") else stem
        if not os.path.exists(os.path.join(STRUCT_DIR, f"{table}.csv")):
            print("нет структуры для", fname); continue
        if os.path.getsize(path) < 16:
            LOAD_LOG.append((stem, 0, 0, "пустой файл")); continue
        cols = struct_cols(table)
        with open(path, encoding="utf-8-sig", errors="replace") as f:
            first = f.readline().strip()
        header = first == ";".join(cols)
        src = read_csv_expr(path, cols, header)
        # доля значений с часовым поясом в каждой колонке времени (по первым 5000 строкам)
        for c in cols:
            if TIME_COL.search(c):
                vals = [v for (v,) in con.sql(f'SELECT "{c}" FROM {src} WHERE "{c}" IS NOT NULL LIMIT 5000').fetchall()]
                if vals:
                    TZ_INFO[f"{table}.{c}"] = round(sum(bool(TZ_RE.search(v)) for v in vals) / len(vals), 3)
        before, t0 = rejects(), time.time()
        con.execute(f"COPY (SELECT {select_list(table, cols)} FROM {src}) TO '{PARQUET}/{stem}.parquet' (FORMAT parquet, COMPRESSION zstd)")
        n = con.sql(f"SELECT count(*) FROM '{PARQUET}/{stem}.parquet'").fetchone()[0]
        LOAD_LOG.append((stem, n, rejects() - before, f"{time.time() - t0:.0f} с, заголовок в файле: {header}"))
        print(f"{stem}: {n} строк, отброшено {LOAD_LOG[-1][2]}, {LOAD_LOG[-1][3]}")
    json.dump({"load_log": LOAD_LOG, "tz_info": TZ_INFO}, open(f"{PARQUET}/_done.json", "w"), ensure_ascii=False)
else:
    meta = json.load(open(f"{PARQUET}/_done.json"))
    LOAD_LOG, TZ_INFO = meta["load_log"], meta["tz_info"]
    print("Parquet уже собран, загрузка пропущена")

In [ ]:
# Представления: одна таблица = один Parquet; месячные логи подключений объединены в conn
for f in sorted(os.listdir(PARQUET)):
    if f.endswith(".parquet") and not f.startswith("user_connection_logs_"):
        con.execute(f"CREATE OR REPLACE VIEW {f[:-8]} AS SELECT * FROM '{PARQUET}/{f}'")
con.execute(f"""CREATE OR REPLACE VIEW conn AS
    SELECT *, regexp_extract(filename, 'user_connection_logs_([0-9_]+)\\.parquet', 1) AS src_file
    FROM read_parquet('{PARQUET}/user_connection_logs_*.parquet', filename = true)""")
print(con.sql("SHOW TABLES").df()["name"].tolist())

## 3. Разведка

Только агрегаты. Весь вывод дублируется в `recon_report.txt` (на Drive, рядом с Parquet) — его и нужно прислать.

In [ ]:
REPORT = []
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)

def out(*a):
    s = " ".join(str(x) for x in a); print(s); REPORT.append(s)

def show(title, sql, n=60):
    out(f"\n## {title}")
    try:
        df = con.sql(sql).df()
        out(df.head(n).to_string(index=False) if len(df) else "(пусто)")
        return df
    except Exception as e:
        out(f"ОШИБКА: {e}")

def save_report():
    with open(f"{BASE}/recon_report.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(REPORT))

out("# Загрузка: таблица, строк, отброшено, комментарий")
for row in LOAD_LOG:
    out("  ", *row)
out("\n# Доля значений с часовым поясом по колонкам времени")
for k, v in sorted(TZ_INFO.items()):
    out(f"   {k}: {v}")

In [ ]:
out("\n# 3.1 Целостность ключей")
show("users.id уникален?", "SELECT count(*) n, count(DISTINCT id) n_distinct FROM users")

def link(name, child, ckey, parent, pkey):
    return show(name, f"""
        SELECT count(*) n, count(c.{ckey}) n_nonnull,
               round(avg(CASE WHEN c.{ckey} IS NULL THEN NULL WHEN p.k IS NOT NULL THEN 1 ELSE 0 END), 4) share_found
        FROM {child} c LEFT JOIN (SELECT DISTINCT {pkey} AS k FROM {parent}) p ON c.{ckey} = p.k""")

for args in [("orders.user_id → users", "orders", "user_id", "users", "id"),
             ("orders.product_id → product", "orders", "product_id", "product", "id"),
             ("transactions.order_id → orders", "transactions", "order_id", "orders", "id"),
             ("subscribe.order_id → orders", "subscribe", "order_id", "orders", "id"),
             ("subscribe.user_id → users", "subscribe", "user_id", "users", "id"),
             ("devices.user_id → users", "devices", "user_id", "users", "id"),
             ("sessions.device_id → devices", "sessions", "device_id", "devices", "id"),
             ("adv_view_history.user_id → users", "adv_view_history", "user_id", "users", "id"),
             ("conn.user_id → users", "conn", "user_id", "users", "id"),
             ("conn.subscribe_id → subscribe", "conn", "subscribe_id", "subscribe", "id"),
             ("conn.node_id → nodes", "conn", "node_id", "nodes", "id"),
             ("saved_payment_methods.user_id → users", "saved_payment_methods", "user_id", "users", "id")]:
    link(*args)
save_report()

In [ ]:
out("\n# 3.2 Часовые пояса (поля без пояса: UTC или Москва?)")
show("subscribe: (expired_time [с поясом → UTC] − created_at [без пояса]) в часах по модулю 24; мода 0 ⇒ без пояса = UTC, мода 21 ⇒ Москва", """
    SELECT ((round(epoch(expired_time - created_at) / 3600)::BIGINT % 24) + 24) % 24 AS hour_mod, count(*) n
    FROM subscribe WHERE expired_time IS NOT NULL AND created_at IS NOT NULL
    GROUP BY 1 ORDER BY n DESC LIMIT 6""")
show("первая сессия [с поясом → UTC] − users.created_at [без пояса], часы; мода 0 ⇒ UTC, мода −3 ⇒ users.created_at в московском времени", """
    WITH fs AS (SELECT user_id, min(created_at) t FROM sessions GROUP BY 1)
    SELECT round(epoch(fs.t - u.created_at) / 3600)::BIGINT diff_h, count(*) n
    FROM fs JOIN users u ON u.id = fs.user_id
    WHERE abs(epoch(fs.t - u.created_at)) < 6 * 3600
    GROUP BY 1 ORDER BY n DESC LIMIT 6""")
save_report()

In [ ]:
out("\n# 3.3 Логи подключений: покрытие и дыры")
show("по месяцам", """
    SELECT strftime(created_at, '%Y-%m') m, count(*) n, count(DISTINCT user_id) users,
           round(avg((subscribe_id IS NULL)::INT), 3) no_sub_share, count(DISTINCT CAST(created_at AS DATE)) n_days
    FROM conn GROUP BY 1 ORDER BY 1""")
daily = con.sql("SELECT CAST(created_at AS DATE) d, count(*) n FROM conn GROUP BY 1 ORDER BY 1").df()
daily["d"] = pd.to_datetime(daily["d"])
full = pd.date_range(daily["d"].min(), daily["d"].max(), freq="D")
missing = sorted(set(full) - set(daily["d"]))
out("дней без единой записи:", len(missing), [d.strftime("%Y-%m-%d") for d in missing[:40]])
daily["med7"] = daily["n"].rolling(15, center=True, min_periods=5).median()
low = daily[daily["n"] < 0.3 * daily["med7"]]
out("дней с объёмом < 30% от скользящей медианы:", len(low))
out(low[["d", "n", "med7"]].head(40).to_string(index=False) if len(low) else "")
show("одинаковые id строк в разных месячных файлах (дубли выгрузки)", """
    SELECT count(*) n_dup_ids FROM (SELECT id FROM conn GROUP BY id HAVING count(*) > 1)""")
show("файл ↔ месяц записи (нет ли записей чужого месяца)", """
    SELECT src_file, strftime(created_at, '%Y-%m') m, count(*) n FROM conn GROUP BY 1, 2 ORDER BY 1, 2""")
save_report()

In [ ]:
out("\n# 3.4 Бесплатный тариф и ноды")
show("продукты", "SELECT id, name, limit_id, amount, currency, created_at FROM product ORDER BY id")
show("периоды", "SELECT * FROM product_periods ORDER BY id")
show("скидки на периоды", "SELECT * FROM product_period_discounts ORDER BY product_id, period_id")
show("country_disabled_products", """
    SELECT c.country_code, c.product_id, p.name, c.created_at FROM country_disabled_products c
    LEFT JOIN product p ON p.id = c.product_id""")
show("ноды: free_pool × auto_select", "SELECT free_pool, auto_select, count(*) n FROM nodes GROUP BY ALL ORDER BY n DESC")
show("топ-5 нод по подключениям в каждом месяце", """
    WITH c AS (SELECT strftime(created_at, '%Y-%m') m, node_id, count(*) n, avg((subscribe_id IS NULL)::INT) no_sub
               FROM conn GROUP BY 1, 2),
         r AS (SELECT *, row_number() OVER (PARTITION BY m ORDER BY n DESC) rk, n / sum(n) OVER (PARTITION BY m) share_n FROM c)
    SELECT r.m, r.rk, substr(r.node_id, 1, 8) node, round(r.share_n, 3) share_n, round(r.no_sub, 3) no_sub,
           nd.free_pool, nd.auto_select, l.country_code
    FROM r LEFT JOIN nodes nd ON nd.id = r.node_id LEFT JOIN locations l ON l.id = nd.location_id
    WHERE r.rk <= 5 ORDER BY r.m, r.rk""")
show("подключения без subscribe_id: есть ли у пользователя в этот момент активная подписка (выборка 3%)", """
    SELECT strftime(c.created_at, '%Y-%m') m, count(*) n,
           round(avg(CASE WHEN EXISTS (SELECT 1 FROM subscribe s WHERE s.user_id = c.user_id
                 AND c.created_at BETWEEN s.created_at AND s.expired_time) THEN 1 ELSE 0 END), 3) share_active_sub
    FROM (SELECT user_id, created_at FROM conn WHERE subscribe_id IS NULL AND random() < 0.03) c
    GROUP BY 1 ORDER BY 1""")
save_report()

In [ ]:
out("\n# 3.5 Реклама")
show("adv_view_history по месяцам", """
    SELECT strftime(created_at, '%Y-%m') m, count(*) n_views, count(DISTINCT user_id) users,
           round(count(*) / count(DISTINCT user_id), 2) per_user
    FROM adv_view_history GROUP BY 1 ORDER BY 1""")
show("просмотров на пользователя в день (по дням, где был хотя бы один)", """
    WITH d AS (SELECT user_id, CAST(created_at AS DATE) d, count(*) n FROM adv_view_history GROUP BY 1, 2)
    SELECT count(*) user_days, quantile_disc(n, [0.5, 0.75, 0.9, 0.95, 0.99]) q50_75_90_95_99, max(n) mx FROM d""")
show("интервал между соседними просмотрами одного пользователя, секунды (дубли записи?)", """
    WITH a AS (SELECT epoch(created_at - lag(created_at) OVER (PARTITION BY user_id ORDER BY created_at)) dt
               FROM adv_view_history)
    SELECT quantile_cont(dt, [0.01, 0.1, 0.5, 0.9]) q01_10_50_90, avg((dt < 5)::INT) share_lt_5s FROM a WHERE dt IS NOT NULL""")
show("mobile_version: платформы", "SELECT platform, count(*) n, min(created_at) first_at, max(created_at) last_at FROM mobile_version GROUP BY 1")
show("mobile_version: ключи настроек", """
    SELECT k, count(*) n FROM (SELECT unnest(json_keys(global_settings)) k FROM mobile_version WHERE json_valid(global_settings))
    GROUP BY 1 ORDER BY n DESC""")
show("mobile_version: флаги рекламы и обязательного обновления по версиям", """
    SELECT platform, CAST(created_at AS DATE) d, version,
           json_extract_string(global_settings, '$.ads.showRewardedAdLimit') rew_limit,
           json_extract_string(ru_settings, '$.ads.showRewardedAdLimit') rew_limit_ru,
           json_extract_string(global_settings, '$.ads.showRewardedAd') rew,
           json_extract_string(ru_settings, '$.ads.showRewardedAd') rew_ru,
           json_extract_string(global_settings, '$.requiredUpdate') req_upd,
           json_extract_string(global_settings, '$.featSub') feat_sub
    FROM mobile_version ORDER BY platform, created_at""", n=200)
save_report()

In [ ]:
out("\n# 3.6 Триал: два источника")
show("users.trial_used_at", "SELECT count(*) users, count(trial_used_at) with_trial, min(trial_used_at) first_at, max(trial_used_at) last_at FROM users")
show("users.trial_used_at по месяцам", """
    SELECT strftime(trial_used_at, '%Y-%m') m, count(*) n FROM users WHERE trial_used_at IS NOT NULL GROUP BY 1 ORDER BY 1""")
show("orders: trial × status", """
    SELECT trial, status, count(*) n, min(created_at) first_at, max(created_at) last_at FROM orders GROUP BY 1, 2 ORDER BY 1, 2""")
show("orders с trial = t: суммы", """
    SELECT amount, status, count(*) n FROM orders WHERE trial = 't' GROUP BY 1, 2 ORDER BY n DESC LIMIT 15""")
show("заказы с триалом по месяцам", """
    SELECT strftime(created_at, '%Y-%m') m, count(*) FILTER (WHERE trial = 't') trial_orders,
           count(*) FILTER (WHERE trial = 't' AND status = 'paid') trial_paid,
           count(*) FILTER (WHERE trial = 'f' AND status = 'paid') regular_paid
    FROM orders GROUP BY 1 ORDER BY 1""")
show("пересечение источников", """
    WITH t AS (SELECT user_id, min(created_at) first_trial FROM orders WHERE trial = 't' AND status = 'paid' GROUP BY 1)
    SELECT count(*) users_with_paid_trial_order, count(u.trial_used_at) of_them_with_trial_used_at,
           quantile_cont(epoch(u.trial_used_at - t.first_trial) / 60, [0.1, 0.5, 0.9]) diff_minutes
    FROM t JOIN users u ON u.id = t.user_id""")
show("trial_used_at есть, а заказа с триалом нет", """
    SELECT count(*) n FROM users u WHERE trial_used_at IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM orders o WHERE o.user_id = u.id AND o.trial = 't')""")
show("длительность подписки по заказу с триалом, дни", """
    SELECT round(epoch(s.expired_time - s.created_at) / 86400, 1) n_days, count(*) n
    FROM subscribe s JOIN orders o ON o.id = s.order_id WHERE o.trial = 't' GROUP BY 1 ORDER BY n DESC LIMIT 10""")
save_report()

In [ ]:
out("\n# 3.7 Заказы и платежи")
show("продукт × статус", """
    SELECT o.product_id, p.name, o.status, count(*) n FROM orders o LEFT JOIN product p ON p.id = o.product_id
    GROUP BY ALL ORDER BY 1, 3""")
show("период × статус", "SELECT period_id, status, count(*) n FROM orders GROUP BY 1, 2 ORDER BY 1, 2")
show("провайдер × статус", """
    SELECT o.provider_id, pp.name, o.status, count(*) n FROM orders o LEFT JOIN payment_providers pp ON pp.id = o.provider_id
    GROUP BY ALL ORDER BY 1, 3""")
show("pre_payment × trial", "SELECT pre_payment, trial, count(*) n FROM orders GROUP BY 1, 2 ORDER BY 1, 2")
show("частые суммы успешных обычных заказов", """
    SELECT product_id, period_id, provider_id, amount, count(*) n FROM orders
    WHERE status = 'paid' AND trial = 'f' GROUP BY ALL ORDER BY n DESC LIMIT 25""")
show("доля неуспешных заказов по месяцам", """
    SELECT strftime(created_at, '%Y-%m') m, count(*) n, round(avg((status = 'failed')::INT), 3) failed,
           round(avg((status = 'pending')::INT), 3) pending FROM orders GROUP BY 1 ORDER BY 1""")
show("transactions: статус", "SELECT status, count(*) n FROM transactions GROUP BY 1 ORDER BY n DESC")
show("transactions: ключи JSON ответа платёжки", """
    SELECT k, count(*) n FROM (SELECT unnest(resp_keys) k FROM transactions) GROUP BY 1 ORDER BY n DESC LIMIT 40""")
show("transactions: статус и paid из JSON", """
    SELECT resp_status, resp_paid, resp_currency, count(*) n FROM transactions GROUP BY ALL ORDER BY n DESC LIMIT 20""")
show("transactions: причины отказа Yookassa", """
    SELECT yk_cancel_reason, yk_cancel_party, count(*) n FROM transactions WHERE yk_cancel_reason IS NOT NULL
    GROUP BY ALL ORDER BY n DESC LIMIT 20""")
show("transactions: причины отказа (другие провайдеры)", """
    SELECT other_reason, count(*) n FROM transactions WHERE other_reason IS NOT NULL GROUP BY 1 ORDER BY n DESC LIMIT 20""")
show("транзакций на заказ", """
    SELECT n_tx, count(*) orders FROM (SELECT order_id, count(*) n_tx FROM transactions GROUP BY 1) GROUP BY 1 ORDER BY 1 LIMIT 10""")
show("есть ли транзакция у заказа, по статусу заказа", """
    SELECT o.status, count(*) n, round(avg((t.order_id IS NOT NULL)::INT), 3) has_tx
    FROM orders o LEFT JOIN (SELECT DISTINCT order_id FROM transactions) t ON t.order_id = o.id GROUP BY 1""")
save_report()

In [ ]:
out("\n# 3.8 Пользователи, устройства, платформы")
show("роли", """
    SELECT role, count(*) n, round(avg((is_was_guest = 't')::INT), 4) was_guest,
           round(avg((deleted_at IS NOT NULL)::INT), 4) deleted, round(avg((trial_used_at IS NOT NULL)::INT), 4) trial_used
    FROM users GROUP BY 1 ORDER BY n DESC""")
show("регистрации по месяцам", """
    SELECT strftime(created_at, '%Y-%m') m, count(*) n, round(avg((role = 'guest')::INT), 3) guest_share
    FROM users GROUP BY 1 ORDER BY 1""")
show("домены почты (почта замаскирована владельцем, домен оставлен)", """
    SELECT email_domain, count(*) n FROM users GROUP BY 1 ORDER BY n DESC LIMIT 15""")
show("devices.brand", "SELECT lower(trim(brand)) brand, count(*) n FROM devices GROUP BY 1 ORDER BY n DESC LIMIT 40")
show("devices: os × client_type", """
    SELECT lower(trim(os)) os, client_type, count(*) n FROM devices GROUP BY 1, 2 ORDER BY n DESC LIMIT 30""")
show("devices: покрытие полей", """
    SELECT count(*) n, count(user_id) with_user, count(device_identifier) with_ident,
           count(*) FILTER (WHERE has_fcm_push_token) fcm, count(*) FILTER (WHERE has_hms_push_token) hms,
           count(*) FILTER (WHERE lower(brand) IN ('huawei', 'honor')) huawei_honor
    FROM devices""")
show("устройств на пользователя", """
    SELECT n_dev, count(*) users FROM (SELECT user_id, count(*) n_dev FROM devices WHERE user_id IS NOT NULL GROUP BY 1)
    GROUP BY 1 ORDER BY 1 LIMIT 12""")
show("пользователей на один device_identifier", """
    SELECT n_users, count(*) identifiers FROM (SELECT device_identifier, count(DISTINCT user_id) n_users FROM devices
    WHERE device_identifier IS NOT NULL GROUP BY 1) GROUP BY 1 ORDER BY 1 LIMIT 12""")
show("external_accounts", """
    SELECT service_name, count(*) n, count(DISTINCT user_id) users, count(DISTINCT account_id) accounts
    FROM external_accounts GROUP BY 1""")
show("saved_payment_methods: карты", """
    SELECT count(*) n, count(card_fp) with_fp, count(DISTINCT card_fp) distinct_cards, count(DISTINCT user_id) users
    FROM saved_payment_methods""")
show("saved_payment_methods: тип карты и страна банка", """
    SELECT card_type, card_country, count(*) n FROM saved_payment_methods GROUP BY ALL ORDER BY n DESC LIMIT 10""")
show("sessions: user_agent", "SELECT user_agent, count(*) n FROM sessions GROUP BY 1 ORDER BY n DESC LIMIT 15")
save_report()

## 4. Разведка клиентских логов

Архивы не распаковываются: читаются прямо из zip. В выводе IP, e-mail и токены маскируются.

In [ ]:
IP_RE = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
MAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
TOKEN_RE = re.compile(r"\b[A-Za-z0-9_\-]{32,}(?:\.[A-Za-z0-9_\-]+)*\b")
UUID_RE = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F-]{27}")
LONGNUM_RE = re.compile(r"\b\d{7,}\b")
HDR_RE = re.compile(r"Running on Os\.(\w+); OS version ([^;]+); App version ([\w.\-+]+)")
NAME_DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})T", re.I)

def mask(s):
    s = IP_RE.sub("<ip>", s); s = MAIL_RE.sub("<email>", s); s = TOKEN_RE.sub("<token>", s)
    s = UUID_RE.sub("<uuid>", s); return LONGNUM_RE.sub("<num>", s)[:200]

def kind(name):
    base = name.replace("\\", "/").split("/")[-1].strip().lower()
    if NAME_DATE_RE.search(base) and base.endswith(".log"): return "app_log"
    if "vpn_access" in base: return "vpn_access"
    if "vpn_error" in base: return "vpn_error"
    if "peepster" in base or "logcat" in base: return "logcat"
    return base.rsplit(".", 1)[-1] if "." in base else "other"

ZIPS = []
for d in sorted(os.listdir(LOG_ROOT)):
    p = os.path.join(LOG_ROOT, d)
    if d.isdigit() and os.path.isdir(p):
        ZIPS += [(d, os.path.join(p, f)) for f in os.listdir(p) if f.lower().endswith(".zip")]
folders = sorted({d for d, _ in ZIPS})
out(f"\n# 4.1 Логи: папок {len(folders)}, архивов {len(ZIPS)}")

user_ids = set(con.sql("SELECT id FROM users").df()["id"])
found = [d for d in folders if d in user_ids]
out(f"номер папки найден среди users.id: {len(found)} из {len(folders)}")
remna = dict(con.sql("SELECT id, remna_short_id FROM users WHERE remna_short_id IS NOT NULL").fetchall())
sha = lambda v: getattr(hashlib, HASHFN)((SALT + v).encode()).hexdigest()
stems = [(d, os.path.basename(p)[:-4]) for d, p in ZIPS]
same_user = sum(remna.get(d) == sha(s) for d, s in stems)
any_user = len({sha(s) for _, s in stems} & set(remna.values()))
out(f"имя архива = remna_short_id того же пользователя: {same_user} из {len(stems)}; совпадает с чьим-либо: {any_user}")
show("кто прислал логи: роль и наличие оплаты", f"""
    SELECT u.role, count(*) n,
           sum(CASE WHEN EXISTS (SELECT 1 FROM orders o WHERE o.user_id = u.id AND o.status = 'paid') THEN 1 ELSE 0 END) with_paid_order
    FROM users u WHERE u.id IN ({",".join("'" + d + "'" for d in found) or "''"}) GROUP BY 1""")
save_report()

In [ ]:
# 4.2 Платформа, версия ОС и приложения из заголовка каждого app_log (читаем первые строки всех файлов)
hdr_rows, kinds = [], collections.Counter()
t0 = time.time()
for d, p in ZIPS:
    try:
        with zipfile.ZipFile(p) as z:
            for i in z.infolist():
                if i.is_dir(): continue
                k = kind(i.filename); kinds[k] += 1
                if k != "app_log": continue
                m_date = NAME_DATE_RE.search(i.filename)
                with z.open(i) as f:
                    for _ in range(5):
                        line = f.readline().decode("utf-8", "replace")
                        m = HDR_RE.search(line)
                        if m:
                            hdr_rows.append((d, m_date.group(1) if m_date else None, *m.groups())); break
    except zipfile.BadZipFile:
        kinds["<битый архив>"] += 1
out(f"\n# 4.2 Типы файлов в архивах ({time.time() - t0:.0f} с):", dict(kinds.most_common()))
H = pd.DataFrame(hdr_rows, columns=["folder", "date", "platform", "os_version", "app_version"])
out(f"app_log с заголовком: {len(H)} из {kinds['app_log']}")
if len(H):
    out("платформа (файлы / папки):")
    out(H.groupby("platform").agg(files=("folder", "size"), folders=("folder", "nunique")).to_string())
    out("топ-20 версий приложения по числу папок:")
    out(H.groupby(["platform", "app_version"])["folder"].nunique().sort_values(ascending=False).head(20).to_string())
    out("файлов по месяцам:")
    out(H.assign(m=H["date"].str[:7]).groupby("m").size().to_string())
save_report()

In [ ]:
# 4.3 Ключевые слова: есть ли в логах лимит, трафик, реклама, покупка (на случайной выборке архивов)
KEYS = {
    "limit": re.compile(r"limit|лимит|quota|exceed", re.I),
    "traffic": re.compile(r"traffic|трафик|uplink|downlink|\bbytes?\b|\b\d+(?:\.\d+)?\s?(?:KB|MB|GB|КБ|МБ|ГБ)\b|usage", re.I),
    "ads": re.compile(r"rewarded|admob|interstitial|\bads?\b|yandex.?ads|banner", re.I),
    "trial": re.compile(r"trial|триал|пробн", re.I),
    "purchase": re.compile(r"purchase|paywall|subscription|подписк|payment|billing|\border\b", re.I),
    "api_dns": re.compile(r"Failed host lookup|No address associated", re.I),
}
ANY = re.compile("|".join(f"(?:{rx.pattern})" for rx in KEYS.values()), re.I)
MAX_LINES_PER_FILE = 300_000
random.seed(SEED)
sample = random.sample(ZIPS, min(LOG_SAMPLE_ZIPS, len(ZIPS)))
hits = collections.Counter(); zips_with = collections.defaultdict(set)
examples = collections.defaultdict(collections.Counter)
lines_total, t0 = 0, time.time()
for d, p in sample:
    try:
        with zipfile.ZipFile(p) as z:
            for i in z.infolist():
                if i.is_dir(): continue
                k = kind(i.filename)
                with z.open(i) as f:
                    for n_line, raw in enumerate(f):
                        if n_line >= MAX_LINES_PER_FILE: break
                        lines_total += 1
                        line = raw.decode("utf-8", "replace")
                        if not ANY.search(line): continue
                        for key, rx in KEYS.items():
                            if rx.search(line):
                                hits[(key, k)] += 1; zips_with[key].add(p)
                                examples[(key, k)][re.sub(r"\d", "9", mask(line.strip()))[:160]] += 1
    except zipfile.BadZipFile:
        pass
out(f"\n# 4.3 Ключевые слова: архивов {len(sample)}, строк {lines_total}, {time.time() - t0:.0f} с")
for key in KEYS:
    out(f"\n[{key}] архивов с упоминанием: {len(zips_with[key])} из {len(sample)}")
    for (kk, k), n in sorted(hits.items(), key=lambda x: -x[1]):
        if kk != key: continue
        out(f"   {k}: {n} строк; частые шаблоны:")
        for t, c in examples[(kk, k)].most_common(4):
            out(f"      {c:>7}  {t}")
# в шаблонах выше цифры заменены на 9; здесь 10 примеров с настоящими числами (IP, почта, токены скрыты)
raw_traffic = []
for d, p in sample[:20]:
    with zipfile.ZipFile(p) as z:
        for i in z.infolist():
            if i.is_dir() or kind(i.filename) != "app_log": continue
            with z.open(i) as f:
                for raw in f:
                    line = raw.decode("utf-8", "replace")
                    if KEYS["traffic"].search(line) or KEYS["limit"].search(line):
                        raw_traffic.append(mask(line.strip()))
                        if len(raw_traffic) >= 10: break
            if len(raw_traffic) >= 10: break
    if len(raw_traffic) >= 10: break
out("\nпримеры строк о трафике/лимите с числами:")
for s in raw_traffic: out("   ", s)
save_report()
out(f"\nОтчёт сохранён: {BASE}/recon_report.txt")